# CardingForums — 00. Reconocimiento del dataset

Este es el primer notebook del caso **CardingForums**. Antes de escribir una sola línea de código de análisis, necesitamos entender QUÉ tenemos entre manos: el contexto criminológico del fenómeno, la estructura técnica del dataset, y las preguntas que vamos a intentar responder.

**¿Por qué empezar con reconocimiento?**  
Porque el análisis sin contexto produce basura con confianza. Si no sabemos qué es un *dump*, qué es un *CVV*, o qué rol cumple un *checker* en el ecosistema del carding, vamos a malinterpretar lo que vemos en los datos. El reconocimiento es el equivalente a leer el brief antes de diseñar.

---

## 1. ¿Qué son los carding forums?

**Carding** es el uso y tráfico de datos robados de tarjetas de crédito. El término engloba todo el ecosistema delictivo que gira alrededor del robo, la validación, la venta y el uso de datos financieros de terceros.

Los **carding forums** son comunidades online — generalmente en la dark web o en dominios .su/.biz/.cc — donde este ecosistema opera de forma más o menos abierta. Funcionan como cualquier foro de internet (vBulletin, en este caso), con subforos temáticos, reputación de usuarios, y sistemas de confianza propios.

### Glosario mínimo

Para entender los datos necesitas conocer estos términos:

| Término | Significado |
|---------|-------------|
| **Dump** | Datos completos de la banda magnética de una tarjeta (Track 1 + Track 2). Sirven para clonar tarjetas físicas. |
| **CVV / CC** | Número de tarjeta + fecha de vencimiento + código CVV. Sirven para compras online sin tarjeta física. |
| **Fullz** | Paquete completo de identidad: nombre, dirección, SSN (número de seguro social), fecha de nacimiento + datos de tarjeta. Los más caros. |
| **Checker** | Servicio (o usuario) que verifica si una tarjeta sigue activa antes de venderla. Un dato "checked" vale más. |
| **Cashout** | Proceso de convertir datos robados en dinero real. El último paso de la cadena. |
| **VBV** | Verified by Visa — tarjetas con esta protección son más difíciles de usar en compras online. |
| **Dump shop** | Tienda automatizada de dumps/CVVs, frecuentemente anunciada en foros. |

### El ecosistema de roles

En estos foros no todos los usuarios hacen lo mismo. Hay roles bastante definidos:

- **Vendedores**: publican lotes de dumps/CVVs con características (banco, país, tipo de tarjeta, precio)
- **Compradores**: adquieren los datos para usarlos directamente o revenderlos
- **Intermediarios / escrow**: actúan como garantes en transacciones entre partes que no se conocen
- **Administradores**: moderan el foro, resuelven disputas, tienen autoridad máxima
- **Lurkers / nuevos**: usuarios con poca actividad, que consumen información sin producir reputación

**Una de las preguntas centrales de nuestro análisis es si podemos inferir estos roles a partir del comportamiento observable en los datos.**

## 2. El dataset

Nuestro dataset consiste en **9 dumps de foros de carding** que abarcan el período 2009–2021. Todos son exportaciones SQL de foros que usaban el software **vBulletin** — un sistema de foros muy popular en esa época.

| Foro | Fecha del dump | Notas |
|------|---------------|-------|
| Carder.su | 2009-02 | Uno de los primeros foros importantes |
| Carding.biz | 2009-11 | |
| Carders.cc | 2010-12 | |
| Cardersplanet.biz | 2010-05 | |
| Carder.pro | 2013-04 | Foro de habla rusa, encoding cp1251 |
| Cardingmafia.ws | 2016-02 | |
| Crdshop.su | 2016-11 | |
| Elitecarders.name | 2016-08 | |
| CardingMafia / Cardmafia.cc | 2021-03 | El dump más reciente |

### ¿Por qué vBulletin?

vBulletin produce exportaciones SQL con un schema muy estandarizado. Eso significa que un solo parser puede leer los 9 foros sin modificaciones. Las tablas principales que nos interesan son:

- `user` — datos de registro de cada usuario (username, email, ICQ, fecha de registro, IP...)
- `post` — cada mensaje publicado en el foro (autor, thread, texto, timestamp)
- `thread` — los hilos de conversación (título, subforo, primer post)
- `pmtext` — mensajes privados (cuando están disponibles en el dump)
- `userfield` — campos adicionales del perfil (campos custom del foro)

## 3. Setup inicial

Antes de explorar los datos, necesitamos cargar las herramientas. Esta celda importa las librerías del curso y las configura para el análisis.

**¿Qué hace cada import?**
- `sys.path.insert(0, "../")` — le dice a Python dónde encontrar el código del curso (la carpeta `src/`)
- `pandas` — la librería principal para trabajar con datos tabulares (como una hoja de cálculo programable)
- `matplotlib` y `seaborn` — para hacer gráficos estáticos
- `plotly` — para gráficos interactivos (puedes hacer zoom, hover, filtrar)
- `load_forum`, `list_forums` — funciones del curso que leen los dumps de SQL
- `DATA_DIR`, `RESULTS_DIR` — rutas a las carpetas de datos y resultados

In [ ]:
import sys
sys.path.insert(0, "../")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from src.utils import list_forums, merge_tables, load_forum, DATA_DIR, RESULTS_DIR

RESULTS_DIR.mkdir(exist_ok=True)
sns.set_theme(style="darkgrid")

CATEGORY = "Carding Forums"
forum_zips = list_forums(CATEGORY)

print(f"Foros disponibles: {len(forum_zips)}")
for p in forum_zips:
    print(f"  {p.name}")

## 4. Exploración del dump: tablas y registros

Vamos a cargar UNO de los foros para inspeccionar la estructura antes de procesar los 9. Elegimos el primero de la lista.

### ¿Qué es `load_forum()`?

`load_forum()` es la función principal del parser del curso. Recibe la ruta a un archivo `.zip` con el dump SQL y devuelve un diccionario donde cada clave es el nombre de una tabla (`"user"`, `"post"`, etc.) y el valor es un DataFrame de pandas con esa tabla.

El parámetro `tables` le dice qué tablas queremos. Si no lo especificamos, intenta cargar todas.

> `load_forum()` vive en `src/utils.py` y auto-detecta el formato del dump (vBulletin/MyBB/IPS/flat) — por eso funciona igual en los 8 vBulletin de este caso y en el flat file de Cardingmafia.ws, sin código distinto. Ver [`src/README.md`](../src/README.md) para el resto de la API de carga.

In [ ]:
# Cargamos el primer foro para inspeccionar su estructura
sample_forum = forum_zips[0]
print(f"Inspeccionando: {sample_forum.name}")

sample_dfs = load_forum(sample_forum)

print(f"\nTablas disponibles en este dump:")
for table_name, df in sample_dfs.items():
    print(f"  {table_name:20s} — {len(df):,} filas, {len(df.columns)} columnas")

### ¿Qué tiene la tabla `user`?

La tabla `user` es la más rica para análisis forense. Cada fila es un usuario registrado. Veamos sus columnas y los primeros registros.

**¿Qué es `df.dtypes`?** Nos muestra el tipo de dato de cada columna: `object` significa texto, `int64` es número entero, `float64` es decimal, `datetime64` es fecha.

In [ ]:
users_sample = sample_dfs.get("user", pd.DataFrame())

print(f"Columnas de la tabla user ({len(users_sample.columns)} en total):")
print(users_sample.dtypes.to_string())
print(f"\nPrimeras 3 filas:")
users_sample.head(3)

### ¿Qué tiene la tabla `post`?

La tabla `post` contiene cada mensaje publicado en el foro. Es la más grande y la que más usaremos para análisis semántico (¿de qué habla cada usuario?).

In [ ]:
posts_sample = sample_dfs.get("post", pd.DataFrame())

print(f"Columnas de la tabla post ({len(posts_sample.columns)} en total):")
print(posts_sample.dtypes.to_string())
print(f"\nPrimeras 3 filas:")
posts_sample[["postid", "threadid", "userid", "dateline", "pagetext"]].head(3)

## 5. Carga completa de los 9 foros

Ahora cargamos todos los foros y combinamos las tablas. La función `merge_tables()` toma la lista de diccionarios (uno por foro) y concatena la misma tabla de todos los foros en un único DataFrame, añadiendo una columna `forum` para saber de dónde viene cada fila.

**Esto puede tardar 1–2 minutos** — estamos leyendo varios cientos de MB de SQL.

> Con 10 foros que combinar, `merge_tables()` (`src/utils.py`) es la función clave de este caso: evita concatenar tabla por tabla a mano y añade sola la columna `forum` de origen. Detalle en [`src/README.md`](../src/README.md).

In [ ]:
from tqdm.notebook import tqdm

all_forums = []
for path in tqdm(forum_zips, desc="Parseando foros"):
    dfs = load_forum(path, tables=["user", "post", "thread", "pmtext"])
    all_forums.append(dfs)

users   = merge_tables(all_forums, "user")
posts   = merge_tables(all_forums, "post")
threads = merge_tables(all_forums, "thread")
pms     = merge_tables(all_forums, "pmtext")

print(f"Total usuarios: {len(users):,}")
print(f"Total posts:    {len(posts):,}")
print(f"Total threads:  {len(threads):,}")
print(f"Total PMs:      {len(pms):,}")

## 6. Distribuciones básicas

Antes de analizar nada en profundidad, necesitamos tener una imagen general de los datos: cuántos usuarios tiene cada foro, cuándo estuvieron activos, cómo se distribuyen los posts.

### Usuarios y posts por foro

Esta tabla resume el tamaño de cada foro. El ratio `posts_per_user` es interesante: un valor alto indica una comunidad más activa (los mismos usuarios vuelven a postear mucho), mientras que un valor bajo puede indicar un foro con muchos registros pero poca participación real.

In [ ]:
summary = pd.DataFrame({
    "usuarios": users.groupby("forum").size(),
    "posts":    posts.groupby("forum").size(),
    "pms":      pms.groupby("forum").size() if len(pms) > 0 else 0,
}).fillna(0).astype(int)

summary["posts_por_usuario"] = (summary["posts"] / summary["usuarios"]).round(1)
summary.sort_values("usuarios", ascending=False)

### Timeline de registros

¿Cuándo se registraron los usuarios? Esto nos da una idea del período de actividad de cada foro. Los picos de registro suelen coincidir con momentos de mayor actividad del ecosistema (brechas grandes, cierre de foros competidores, etc.).

**¿Qué es `dt.year`?** La columna `joindate` es una fecha. `.dt` nos permite acceder a sus componentes (año, mes, día, hora). `.dt.year` extrae el año de cada registro.

In [ ]:
# Convertir joindate a datetime si no lo es ya
users["joindate"] = pd.to_datetime(users["joindate"], errors="coerce", unit="s", utc=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Registros por año
users["joindate"].dropna().dt.year.value_counts().sort_index().plot(
    kind="bar", ax=axes[0], title="Registros por año", color="steelblue"
)
axes[0].set_xlabel("Año")
axes[0].set_ylabel("Usuarios registrados")

# Distribución de posts por usuario (con cap en 500 para que se vea bien)
users["posts"] = pd.to_numeric(users.get("posts", pd.Series(dtype=float)), errors="coerce")
users["posts"].clip(upper=500).dropna().plot(
    kind="hist", bins=50, ax=axes[1],
    title="Distribución de posts por usuario (cap 500)",
    color="steelblue", logy=True
)
axes[1].set_xlabel("Número de posts")
axes[1].set_ylabel("Cantidad de usuarios (escala log)")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "00_user_overview.png", dpi=150)
plt.show()

print("\nNota sobre la escala logarítmica en el eje Y:")
print("La mayoría de los usuarios tiene muy pocos posts (cola larga a la derecha).")
print("Esto es típico en foros: pocos usuarios muy activos, muchos casi inactivos.")

### Actividad temporal: posts por mes

La distribución temporal de posts nos muestra cuándo estuvieron activos estos foros. Podemos ver picos de actividad y períodos de silencio.

**¿Qué es `resample('ME')`?** Toma una serie de fechas y la agrupa por mes (Month End). Es el equivalente a un pivot mensual.

In [ ]:
posts["dateline"] = pd.to_datetime(posts["dateline"], errors="coerce", unit="s", utc=True)

# Filtrar fechas razonables (entre 2005 y 2025)
posts_dated = posts[
    posts["dateline"].between(pd.Timestamp("2005-01-01", tz="UTC"), pd.Timestamp("2025-01-01", tz="UTC"))
].copy()

monthly_activity = posts_dated.set_index("dateline").resample("ME").size()

fig = px.line(
    x=monthly_activity.index,
    y=monthly_activity.values,
    title="Actividad mensual en todos los foros (posts)",
    labels={"x": "Fecha", "y": "Posts publicados"}
)
fig.show()
print(f"\nRango temporal cubierto: {posts_dated['dateline'].min().date()} → {posts_dated['dateline'].max().date()}")

### Subforos: ¿cómo está organizado el contenido?

Los foros de carding tienen subforos temáticos que revelan la estructura del mercado. Un subforo llamado "Dumps" es diferente a uno llamado "General Discussion". Esta distribución nos da una primera pista sobre qué tipo de actividad domina en cada foro.

In [ ]:
# Unir posts con threads para obtener el subforo (forumid → forum name)
# Si la tabla thread tiene forumid y title, podemos ver los subforos más activos
if "forumid" in threads.columns:
    # Posts por foro (usando forumid del thread)
    posts_with_forum = posts.merge(
        threads[["threadid", "forumid", "forum"]],
        on=["threadid", "forum"],
        how="left"
    )
    subforum_counts = posts_with_forum.groupby(["forum", "forumid"]).size().reset_index(name="posts")
    print("Posts por subforo (top 20):")
    print(subforum_counts.sort_values("posts", ascending=False).head(20).to_string(index=False))
else:
    print("Columna forumid no disponible en threads. Mostrando distribución de threads por foro.")
    print(threads.groupby("forum").size().sort_values(ascending=False).to_string())

## 7. Validación de calidad del dataset

Antes de analizar, necesitamos identificar problemas en los datos. Los dumps de SQL de foros tienen problemas típicos que hay que conocer antes de sacar conclusiones.

### Problemas conocidos en dumps de vBulletin

1. **Timestamps epoch-0**: En Unix, el timestamp `0` equivale al 1 de enero de 1970. Aparece en registros donde la fecha no fue registrada o fue borrada. Hay que filtrarlos.
2. **Posts vacíos**: Posts con `pagetext` vacío o solo espacios — posts borrados por el moderador, pero cuya estructura de BD quedó.
3. **UserIDs nulos**: Puede haber posts de usuarios que ya no existen en la tabla `user` (fueron baneados y borrados).
4. **Encodings**: Carder.pro usa cp1251 (cirílico). El parser lo maneja, pero podemos verificar.

In [ ]:
print("=" * 50)
print("VALIDACIÓN DE CALIDAD — TABLA users")
print("=" * 50)

# Timestamps epoch-0 en joindate
users["joindate"] = pd.to_datetime(users["joindate"], errors="coerce", unit="s", utc=True)
epoch_zero_users = (users["joindate"] < pd.Timestamp("2000-01-01", tz="UTC")).sum()
null_joindate = users["joindate"].isna().sum()
print(f"  Usuarios con joindate < año 2000 (epoch-0 o inválidos): {epoch_zero_users:,}")
print(f"  Usuarios con joindate nulo: {null_joindate:,}")

# Usernames vacíos
empty_usernames = users["username"].replace("", pd.NA).isna().sum()
print(f"  Usernames vacíos o nulos: {empty_usernames:,}")

print("\n" + "=" * 50)
print("VALIDACIÓN DE CALIDAD — TABLA posts")
print("=" * 50)

# Posts con timestamp epoch-0
posts["dateline"] = pd.to_datetime(posts["dateline"], errors="coerce", unit="s", utc=True)
epoch_zero_posts = (posts["dateline"] < pd.Timestamp("2000-01-01", tz="UTC")).sum()
null_dateline = posts["dateline"].isna().sum()
print(f"  Posts con dateline < año 2000: {epoch_zero_posts:,}")
print(f"  Posts con dateline nulo: {null_dateline:,}")

# Posts vacíos
if "pagetext" in posts.columns:
    empty_posts = posts["pagetext"].replace("", pd.NA).isna().sum()
    print(f"  Posts con texto vacío: {empty_posts:,}")
    very_short = (posts["pagetext"].str.len() < 5).sum()
    print(f"  Posts con menos de 5 caracteres: {very_short:,}")

# UserIDs en posts que no existen en users
user_ids_known = set(pd.to_numeric(users["userid"], errors="coerce").dropna())
post_userids = pd.to_numeric(posts["userid"], errors="coerce")
orphan_posts = (~post_userids.isin(user_ids_known) & post_userids.notna()).sum()
print(f"  Posts de usuarios no encontrados en tabla user: {orphan_posts:,}")

In [ ]:
# Resumen visual de completitud por columna en la tabla users
# Esto nos dice qué información tenemos disponible para análisis
contact_cols = ["email", "icq", "skype", "aim", "yahoo", "msn", "homepage", "ipaddress"]
available = [c for c in contact_cols if c in users.columns]

fill_rate = (
    users[available]
    .replace("", pd.NA)
    .notna()
    .mean()
    .sort_values(ascending=True)
    .mul(100)
)

fill_rate.plot(kind="barh", figsize=(8, 4), title="% de usuarios con cada campo de contacto completado")
plt.xlabel("% de usuarios")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "00_contact_fill_rate.png", dpi=150)
plt.show()

print("\nInterpretación: los campos con mayor tasa de completitud son los más útiles")
print("para correlación entre foros. ICQ era el identificador más reutilizado.")

## 8. Actores más activos: primer vistazo

¿Quiénes son los usuarios con más posts? No son necesariamente los más importantes (un admin puede tener pocos posts pero mucho poder), pero el volumen de actividad es una señal de presencia sostenida.

Estos usuarios van a ser los candidatos principales para el **perfilado de actores** que haremos en el notebook 03.

In [ ]:
# Top 20 usuarios por número de posts
# La columna 'posts' en la tabla user es el contador total del foro
top_posters = (
    users[["forum", "username", "posts", "email", "icq", "joindate"]]
    .assign(posts=lambda df: pd.to_numeric(df["posts"], errors="coerce"))
    .sort_values("posts", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

print("Top 20 usuarios por número de posts:")
top_posters

## 9. Preguntas de investigación

Con este primer panorama del dataset, podemos formular las **preguntas de investigación** que guiarán el resto del análisis.

---

### Pregunta 1: ¿Quiénes son los actores más activos?

Volumen de posts, frecuencia de participación, presencia temporal sostenida. Los usuarios más activos son los más visibles en el ecosistema.

> **Respondida en**: notebook `02_analisis_estructural.ipynb` (ranking de centralidad) y `03_analisis_semantico.ipynb` (perfilado de actores)

---

### Pregunta 2: ¿Hay roles diferenciados — vendedores, compradores, intermediarios?

¿Podemos inferir el rol de un usuario a partir de lo que escribe, con quién interactúa, y en qué subforos participa?

> **Respondida en**: notebook `03_analisis_semantico.ipynb` (NER + perfilado con LLM)

---

### Pregunta 3: ¿Cómo se organiza la red de confianza?

¿Hay usuarios que actúan como puentes entre comunidades? ¿Existen cliques cerrados de confianza? ¿Quién tiene alta betweenness centrality?

> **Respondida en**: notebook `02_analisis_estructural.ipynb` (análisis de red)

---

### Pregunta 4: ¿Hay usuarios que aparecen en múltiples foros?

Las mismas personas operando en distintos foros bajo el mismo (o diferente) username. Correlación por email, ICQ, estilo de escritura.

> **Respondida en**: notebook `01_ingenieria_datos.ipynb` (correlación de campos) y `03_analisis_semantico.ipynb` (similitud de embeddings)

---

### Pregunta 5: ¿Qué se vende en cada subforo?

¿Los subforos tienen especializaciones claras? ¿Hay vocabulario diferenciado?

> **Respondida en**: notebook `02_analisis_estructural.ipynb` (TF-IDF por subforo)

---

Estas preguntas son las que vamos a responder con evidencia en los próximos notebooks. Lo importante: **nunca afirmamos algo que no podamos sostener con datos**. Cuando la evidencia es débil, lo decimos.

In [ ]:
# Guardamos un resumen del reconocimiento para usar en los siguientes notebooks
reconocimiento_summary = {
    "total_forums": len(forum_zips),
    "total_users": len(users),
    "total_posts": len(posts),
    "total_threads": len(threads),
    "total_pms": len(pms),
    "date_range": {
        "min": str(posts["dateline"].min()),
        "max": str(posts["dateline"].max())
    }
}

import json
(RESULTS_DIR / "00_reconocimiento_summary.json").write_text(
    json.dumps(reconocimiento_summary, indent=2, ensure_ascii=False)
)

print("=" * 50)
print("RESUMEN DEL RECONOCIMIENTO")
print("=" * 50)
for k, v in reconocimiento_summary.items():
    print(f"  {k}: {v}")

print("\n→ Resumen guardado en results/00_reconocimiento_summary.json")
print("→ Siguiente paso: 01_ingenieria_datos.ipynb")